# Model Selection and Encoding Experiments
In this notebook we evaluate different machine learning models and encoding strategies for the rent price prediction task.

## Imports and Setup

In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

%load_ext autoreload
%autoreload all
from src.models.pipeline_utils import evaluate_models

## Data Preparation

In [2]:
data = pd.read_csv('../data/processed/train_fe.csv')
black_list = ['id', 'price', 'price_bin', 'residential', 'neighborhood',
              'neighborhood_1', 'neighborhood_2', 'neighborhood_3',
              'neighborhood_4', 'neighborhood_5', 'price_bin',
              'description', 'detail']
num_feats = [col for col in data.select_dtypes(exclude='object').columns if col not in black_list]
cat_feats = [col for col in data.select_dtypes(include='object').columns if col not in black_list]
feats = [col for col in data.columns if col not in black_list]
X = data[feats]
y = data['price']

## Model Configurations

In [3]:
xgb_params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'rmsle',
    'seed': 7,
    'tree_method': 'hist',
    'grow_policy': 'lossguide',
}

lgb_params = {
    'random_state': 7,
    'verbose': 0,
    'force_col_wise': True,
}

MODELS = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(random_state=7),
    'Lasso': Lasso(random_state=7),
    'ElasticNet': ElasticNet(random_state=7),
    'KNN': KNeighborsRegressor(),
    'SVR': SVR(),
    'Decision Tree': DecisionTreeRegressor(random_state=7),
    'Extra Trees': ExtraTreesRegressor(random_state=7, n_jobs=-1),
    'Random Forest': RandomForestRegressor(random_state=7, n_jobs=-1),
    'XGBoost': XGBRegressor(**xgb_params),
    'LightGBM': LGBMRegressor(**lgb_params)
}

## Numerical Features Only

In [4]:
results_dict = {}

In [6]:
pipeline_config = {'numerical only': {
    'numerical_features': num_feats,
    'encoding_strategy': 'none'
}}

df_results = evaluate_models(MODELS, X[num_feats], y, pipeline_config, results_dict)


Linear Regression
[Fold 0] train_rmsle: 0.3097, val_rmsle: 0.3129
[Fold 1] train_rmsle: 0.3091, val_rmsle: 0.3149
[Fold 2] train_rmsle: 0.3084, val_rmsle: 0.3167
RMSLE: 0.3148 ± 0.0016

Ridge
[Fold 0] train_rmsle: 0.3097, val_rmsle: 0.3129
[Fold 1] train_rmsle: 0.3091, val_rmsle: 0.3149
[Fold 2] train_rmsle: 0.3084, val_rmsle: 0.3167
RMSLE: 0.3148 ± 0.0016

Lasso
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

ElasticNet
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

KNN
[Fold 0] train_rmsle: 0.3115, val_rmsle: 0.3865
[Fold 1] train_rmsle: 0.3158, val_rmsle: 0.3780
[Fold 2] train_rmsle: 0.3104, val_rmsle: 0.3903
RMSLE: 0.3850 ± 0.0051

SVR
[Fold 0] train_rmsle: 0.1704, val_rmsle: 0.2912
[Fold 1] train_rmsle: 0.1716, val_rmsle: 0.3077
[Fold 2] train

In [7]:
df_results

,model,RMSLE
10,numerical only LightGBM,0.2505
9,numerical only XGBoost,0.2543
7,numerical only Extra Trees,0.2630
8,numerical only Random Forest,0.2631
5,numerical only SVR,0.2999
1,numerical only Ridge,0.3148
0,numerical only Linear Regression,0.3148
6,numerical only Decision Tree,0.3755
4,numerical only KNN,0.3850
2,numerical only Lasso,0.7359


**Compare with baseline:**
- Boosting models improved significantly after feature engineering:
    - **XGBoost:** 0.286 → 0.254
    - **LightGBM:** 0.290 → 0.251
- Tree ensembles also improved:
    - **Random Forest:** 0.293 → 0.263
    - **Extra Trees:** 0.301 → 0.263
- Linear models improved dramatically:
    - **Linear Regression:** 0.485 → 0.315
    - **Ridge:** 0.315
- **KNN**, **Decision Tree**, **Lasso**, and **ElasticNet** consistently underperformed (RMSLE > 0.315) and are not recommended for further tuning.

Given these results, future experiments should concentrate on LightGBM and XGBoost as primary models, with Ridge, Linear Regression, and SVR serving as complementary baselines. Additional experiments with categorical feature encoding (one-hot, ordinal, and combined approaches) are expected to further enhance the performance of boosting and linear models.



## Encoding Experiments

### OneHot Encoding

In [8]:
pipeline_config = {'onehot encoding': {
    'encoding_strategy': 'onehot',
    'numerical_features': num_feats,
    'categorical_features': cat_feats
}}

df_results = evaluate_models(MODELS, X, y, pipeline_config, results_dict)


Linear Regression
[Fold 0] train_rmsle: 0.2695, val_rmsle: 0.2737
[Fold 1] train_rmsle: 0.2661, val_rmsle: 0.2811
[Fold 2] train_rmsle: 0.2681, val_rmsle: 0.2776
RMSLE: 0.2774 ± 0.0030

Ridge
[Fold 0] train_rmsle: 0.2695, val_rmsle: 0.2735
[Fold 1] train_rmsle: 0.2662, val_rmsle: 0.2809
[Fold 2] train_rmsle: 0.2681, val_rmsle: 0.2776
RMSLE: 0.2773 ± 0.0030

Lasso
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

ElasticNet
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

KNN
[Fold 0] train_rmsle: 0.3011, val_rmsle: 0.3714
[Fold 1] train_rmsle: 0.3020, val_rmsle: 0.3648
[Fold 2] train_rmsle: 0.2963, val_rmsle: 0.3752
RMSLE: 0.3705 ± 0.0043

SVR
[Fold 0] train_rmsle: 0.1486, val_rmsle: 0.2708
[Fold 1] train_rmsle: 0.1460, val_rmsle: 0.2897
[Fold 2] train

In [9]:
df_results

,model,RMSLE
21,onehot encoding LightGBM,0.2471
10,numerical only LightGBM,0.2505
20,onehot encoding XGBoost,0.2522
9,numerical only XGBoost,0.2543
18,onehot encoding Extra Trees,0.2544
19,onehot encoding Random Forest,0.2587
7,numerical only Extra Trees,0.2630
8,numerical only Random Forest,0.2631
12,onehot encoding Ridge,0.2773
11,onehot encoding Linear Regression,0.2774


One-hot encodong for categorical features increase performace for all models.

### Ordinal Encoding

In [10]:
pipeline_config = {'ordinal encoding': {
    'encoding_strategy': 'ordinal',
    'numerical_features': num_feats,
    'categorical_features': cat_feats
}}

df_results = evaluate_models(MODELS, X, y, pipeline_config, results_dict)


Linear Regression
[Fold 0] train_rmsle: 0.2955, val_rmsle: 0.2982
[Fold 1] train_rmsle: 0.2933, val_rmsle: 0.3034
[Fold 2] train_rmsle: 0.2950, val_rmsle: 0.3004
RMSLE: 0.3007 ± 0.0021

Ridge
[Fold 0] train_rmsle: 0.2955, val_rmsle: 0.2982
[Fold 1] train_rmsle: 0.2933, val_rmsle: 0.3034
[Fold 2] train_rmsle: 0.2950, val_rmsle: 0.3004
RMSLE: 0.3007 ± 0.0021

Lasso
[Fold 0] train_rmsle: 0.7014, val_rmsle: 0.6953
[Fold 1] train_rmsle: 0.6995, val_rmsle: 0.7005
[Fold 2] train_rmsle: 0.6978, val_rmsle: 0.7043
RMSLE: 0.7000 ± 0.0037

ElasticNet
[Fold 0] train_rmsle: 0.6754, val_rmsle: 0.6698
[Fold 1] train_rmsle: 0.6735, val_rmsle: 0.6750
[Fold 2] train_rmsle: 0.6726, val_rmsle: 0.6779
RMSLE: 0.6742 ± 0.0034

KNN
[Fold 0] train_rmsle: 0.3017, val_rmsle: 0.3869
[Fold 1] train_rmsle: 0.3046, val_rmsle: 0.3724
[Fold 2] train_rmsle: 0.3052, val_rmsle: 0.3714
RMSLE: 0.3769 ± 0.0071

SVR
[Fold 0] train_rmsle: 0.2529, val_rmsle: 0.2747
[Fold 1] train_rmsle: 0.2502, val_rmsle: 0.2806
[Fold 2] train

In [11]:
df_results

,model,RMSLE
21,onehot encoding LightGBM,0.2471
32,ordinal encoding LightGBM,0.2471
10,numerical only LightGBM,0.2505
31,ordinal encoding XGBoost,0.2507
20,onehot encoding XGBoost,0.2522
9,numerical only XGBoost,0.2543
18,onehot encoding Extra Trees,0.2544
29,ordinal encoding Extra Trees,0.2554
30,ordinal encoding Random Forest,0.2582
19,onehot encoding Random Forest,0.2587


- Gradient boosting models **(LightGBM and XGBoost)** achieved the best performance with **RMSLE ≈ 0.247–0.252**, outperforming linear, tree-based, and kernel methods. 
- The choice between one-hot and ordinal encoding had little impact on boosting and tree-based models, while  for Linear Regression and Ridge one-hot encoding performed better.

### Combine Ordinal and OneHot Encoding

In [ ]:
cat_feats

['subway',
 'district',
 'floor_binned',
 'num_storeys_binned',
 'residential_top',
 'neighborhood_top',
 'district_price_tier',
 'subway_line']

In [12]:
pipeline_config = {'mixed encoding': {
    'encoding_strategy': 'mixed',
    'numerical_features': num_feats,
    'categorical_features': cat_feats,
    'onehot_features': [
        'subway', 'district', 'residential_top',
        'neighborhood_top', 'subway_line'
    ],
    'ordinal_categories': {
        'floor_binned': ['1', '2-5', '6-10', '11-15', '16-20', '21-25', '26+'],
        'num_storeys_binned': ['1-5', '6-10', '11-15', '16-20', '21-30', '31+'],
        'district_price_tier': ['tier_1', 'tier_2', 'tier_3']
    }
}}

df_results = evaluate_models(MODELS, X, y, pipeline_config, results_dict)


Linear Regression
[Fold 0] train_rmsle: 0.2698, val_rmsle: 0.2738
[Fold 1] train_rmsle: 0.2667, val_rmsle: 0.2806
[Fold 2] train_rmsle: 0.2682, val_rmsle: 0.2778
RMSLE: 0.2774 ± 0.0028

Ridge
[Fold 0] train_rmsle: 0.2698, val_rmsle: 0.2737
[Fold 1] train_rmsle: 0.2667, val_rmsle: 0.2804
[Fold 2] train_rmsle: 0.2683, val_rmsle: 0.2778
RMSLE: 0.2773 ± 0.0028

Lasso
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

ElasticNet
[Fold 0] train_rmsle: 0.7376, val_rmsle: 0.7320
[Fold 1] train_rmsle: 0.7351, val_rmsle: 0.7373
[Fold 2] train_rmsle: 0.7344, val_rmsle: 0.7385
RMSLE: 0.7359 ± 0.0028

KNN
[Fold 0] train_rmsle: 0.3007, val_rmsle: 0.3700
[Fold 1] train_rmsle: 0.3010, val_rmsle: 0.3669
[Fold 2] train_rmsle: 0.2957, val_rmsle: 0.3708
RMSLE: 0.3692 ± 0.0017

SVR
[Fold 0] train_rmsle: 0.1588, val_rmsle: 0.2677
[Fold 1] train_rmsle: 0.1573, val_rmsle: 0.2837
[Fold 2] train

In [13]:
df_results

,model,RMSLE
43,mixed encoding LightGBM,0.2471
21,onehot encoding LightGBM,0.2471
32,ordinal encoding LightGBM,0.2471
42,mixed encoding XGBoost,0.2502
10,numerical only LightGBM,0.2505
31,ordinal encoding XGBoost,0.2507
40,mixed encoding Extra Trees,0.2522
20,onehot encoding XGBoost,0.2522
9,numerical only XGBoost,0.2543
18,onehot encoding Extra Trees,0.2544


The experiments confirm that **boosting models (LightGBM, XGBoost)** consistently deliver the best performance across all encoding strategies.
- **LightGBM** achieved the lowest RMSLE of **0.2471**, with no meaningful difference betweens all encoding strategies (mixed, one-hot, ordinal), showing that its results are robust to preprocessing choices.
- **XGBoost** closely followed (best RMSLE = **0.2502** with mixed encoding), slightly behind LightGBM but still outperforming other model families.
- **Extra Trees** achieved the lowest RMSLE of **0.2522** with mixed encoding, that slightly outperforming XGBoost with only numerical features.
**Random Forest** achieved moderate results (RMSLE around **0.258**), showing limited gains from encoding strategies, though behind boosting.
- **Linear models (Ridge, Linear Regression)** and **SVR** stabilized around **0.276–0.300**, performing reasonably well but not competitive with boosting and ensemble models.
- Simpler models **(Decision Tree, KNN)** underperform (**0.365–0.385**), indicating poor suitability for this task.
- Regularized linear models **(Lasso, ElasticNet)** are the weakest, stuck at baseline-level performance (**0.7359**).

**Conclusion**:
- The best-performing and most stable approach is **LightGBM** with categorical encoding (any variant), achieving RMSLE = **0.2471**. Further tuning and feature engineering should prioritize boosting models, as encoding choice does not significantly affect their performance, while other models can serve as comparative baselines only.